In [1]:
%%capture
# Install evaluation dependencies.
# vllm and lxml are assumed to be installed from Model_1_inference.ipynb.
# If running this notebook standalone, uncomment the line below:
!pip install -q vllm lxml
!pip install -q vllm --upgrade
!pip install -q rouge-score sentence-transformers bert-score

In [2]:
# =============================================
# CONFIGURATION — adjust before running
# =============================================
MODEL_ID           = "openai/gpt-oss-20b"
HF_DATASET         = "librocubic/model1-annotation-dataset"
GPU_UTIL           = 0.90
MAX_MODEL_LEN      = 16384
DTYPE              = "bfloat16"
MAX_NEW_TOKENS     = 2000
SUMMARY_WORD_LIMIT = 50

OUTPUT_JSONL = "model1_predictions.jsonl"
RESULTS_JSON = "model1_eval_results.json"

In [4]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.environ["HF_HOME"] = "/content/drive/MyDrive/huggingface_cache"

Mounted at /content/drive


In [5]:
from datasets import load_dataset
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

print(f"Loading evaluation dataset from {HF_DATASET}...")
eval_dataset = load_dataset(HF_DATASET, split="test", token=hf_token)
print(f"Loaded {len(eval_dataset)} samples from test split")

# Sanity check
ex = eval_dataset[0]
print(f"\nExample output: {ex['output']}")

Loading evaluation dataset from librocubic/model1-annotation-dataset...
Loaded 125 samples from test split

Example output: {"annotation": "Attach to existing screen session (screen -dr), switching to full-screen and showing an installed packages listing.", "depth": 0}


In [6]:
import json
import re
from vllm import LLM, SamplingParams
from tqdm.notebook import tqdm

# Load model
print(f"Loading {MODEL_ID}...")
llm = LLM(
    model=MODEL_ID,
    gpu_memory_utilization=GPU_UTIL,
    max_model_len=MAX_MODEL_LEN,
    dtype=DTYPE,
    trust_remote_code=True,
)
tokenizer = llm.get_tokenizer()
sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=MAX_NEW_TOKENS,
    repetition_penalty=1.2,
)
print("Model ready.\n")

def parse_output(text: str) -> dict | None:
    """Strip <think> tags and extract first valid JSON with annotation+depth."""
    if "</think>" in text:
        text = text.split("</think>")[-1].strip()
    dec = json.JSONDecoder()
    i = 0
    while True:
        start = text.find("{", i)
        if start == -1:
            break
        try:
            obj, end = dec.raw_decode(text, start)
            if isinstance(obj, dict) and "annotation" in obj and "depth" in obj:
                return {"annotation": str(obj["annotation"]), "depth": int(obj["depth"])}
            i = end
        except Exception:
            i = start + 1
    return None

# Run inference on every sample in the HF test set
print(f"Running inference on {len(eval_dataset)} samples...")
predictions = {}
parse_errors = 0

for i, sample in enumerate(tqdm(eval_dataset)):
    # The HF dataset stores instruction (static) and input (dynamic) separately
    full_prompt = sample["instruction"] + "\n\n" + sample["input"]
    messages = [{"role": "user", "content": full_prompt}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    output = llm.generate([prompt], sampling_params)
    raw_text = output[0].outputs[0].text.strip()

    parsed = parse_output(raw_text)
    if parsed:
        predictions[i] = parsed
    else:
        parse_errors += 1
        print(f"  WARNING: could not parse output for sample {i}: {raw_text[:100]}")

print(f"\nInference complete. Parsed {len(predictions)}/{len(eval_dataset)} samples.")
if parse_errors:
    print(f"  ({parse_errors} samples failed to parse)")

# Save predictions
with open(OUTPUT_JSONL, "w") as f:
    for idx, pred in predictions.items():
        f.write(json.dumps({"idx": idx, **pred}) + "\n")
print(f"Predictions saved to {OUTPUT_JSONL}")

Loading openai/gpt-oss-20b...
INFO 04-01 00:07:32 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'max_model_len': 16384, 'disable_log_stats': True, 'model': 'openai/gpt-oss-20b'}


config.json: 0.00B [00:00, ?B/s]

INFO 04-01 00:07:56 [model.py:533] Resolved architecture: GptOssForCausalLM


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Parse safetensors files:   0%|          | 0/3 [00:00<?, ?it/s]

...

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Inference complete. Parsed 125/125 samples.
Predictions saved to model1_predictions.jsonl


In [7]:
import json
import os
from rouge_score import rouge_scorer as rouge_module
from sentence_transformers import SentenceTransformer, util as st_util
from bert_score import score as bert_score_fn


def load_gt_from_dataset(dataset):
    """Extract ground-truth annotation/depth pairs from the HF dataset."""
    gt = []
    for sample in dataset:
        parsed = json.loads(sample["output"])
        gt.append({
            "annotation": parsed["annotation"],
            "depth": int(parsed["depth"]),
        })
    return gt


def load_preds(path):
    """Load JSONL predictions from inference output."""
    preds = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            r = json.loads(line)
            preds[r["idx"]] = {
                "annotation": r.get("summary", r.get("annotation", "")),
                "depth": r["depth"],
            }
    return preds


gt_list   = load_gt_from_dataset(eval_dataset)
pred_dict = load_preds(OUTPUT_JSONL)

n_gt      = len(gt_list)
n_pred    = len(pred_dict)
matched   = [(i, gt_list[i], pred_dict[i]) for i in range(n_gt) if i in pred_dict]
n_matched = len(matched)

print(f"GT events:   {n_gt}")
print(f"Predictions: {n_pred}")
print(f"Matched:     {n_matched}")

if n_matched == 0:
    raise ValueError(
        "No matched events. Check that inference completed successfully."
    )

# -- Depth accuracy ----
depth_correct = sum(1 for _, gt, pred in matched if gt["depth"] == pred["depth"])
depth_acc     = depth_correct / n_matched

# -- Text metrics ----
gt_anns   = [gt["annotation"]   for _, gt, _    in matched]
pred_anns = [pred["annotation"] for _, _,  pred in matched]

# ROUGE-L
rscorer      = rouge_module.RougeScorer(["rougeL"], use_stemmer=True)
rouge_scores = [rscorer.score(g, p)["rougeL"].fmeasure for g, p in zip(gt_anns, pred_anns)]
mean_rouge   = sum(rouge_scores) / len(rouge_scores)

# Cosine similarity (sentence-transformers)
st_model  = SentenceTransformer("all-MiniLM-L6-v2")
gt_embs   = st_model.encode(gt_anns,   convert_to_tensor=True)
pred_embs = st_model.encode(pred_anns, convert_to_tensor=True)
cos_sims  = st_util.cos_sim(gt_embs, pred_embs).diag().tolist()
mean_cos  = sum(cos_sims) / len(cos_sims)

# BERTScore
P, R, F1  = bert_score_fn(pred_anns, gt_anns, lang="en", verbose=False)
mean_bert = F1.mean().item()

# -- Print summary ----
print(f"\n{'=' * 54}")
print(f"Model 1 Evaluation  —  {HF_DATASET}")
print(f"{'=' * 54}")
print(f"  Events evaluated:   {n_matched}/{n_gt}")
print(f"  Depth Accuracy:     {depth_acc:.3f}  ({depth_correct}/{n_matched} exact matches)")
print(f"  ROUGE-L:            {mean_rouge:.3f}")
print(f"  Cosine Similarity:  {mean_cos:.3f}")
print(f"  BERTScore F1:       {mean_bert:.3f}")
print(f"{'=' * 54}")

GT events:   125
Predictions: 125
Matched:     125


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Model 1 Evaluation  —  librocubic/model1-annotation-dataset
  Events evaluated:   125/125
  Depth Accuracy:     0.592  (74/125 exact matches)
  ROUGE-L:            0.292
  Cosine Similarity:  0.506
  BERTScore F1:       0.880


In [10]:
# ── Consolidated Summary ──────────────────────────────────────────────────────
import numpy as np

# Depth breakdown by class
gt_depths   = [gt["depth"] for _, gt, _ in matched]
pred_depths = [pred["depth"] for _, _, pred in matched]
depth_classes = sorted(set(gt_depths))

print(f"{'=' * 60}")
print(f"  MODEL 1 EVALUATION SUMMARY  —  {n_matched} samples")
print(f"{'=' * 60}")
print(f"  Depth Accuracy:       {depth_acc:.3f}  ({depth_correct}/{n_matched})")
print(f"  ROUGE-L:              {mean_rouge:.3f}")
print(f"  Cosine Similarity:    {mean_cos:.3f}")
print(f"  BERTScore F1:         {mean_bert:.3f}")

# Per-depth-class accuracy
print(f"\n  Depth Accuracy by Class:")
for dc in depth_classes:
    total = sum(1 for g in gt_depths if g == dc)
    correct = sum(1 for g, p in zip(gt_depths, pred_depths) if g == dc and g == p)
    label = f"depth={dc:+d}" if dc != 0 else "depth= 0"
    print(f"    {label}:  {correct/total:.3f}  ({correct}/{total})")

# Text metric distributions
print(f"\n  ROUGE-L distribution:")
print(f"    min={min(rouge_scores):.3f}  median={np.median(rouge_scores):.3f}  max={max(rouge_scores):.3f}")
print(f"  Cosine Sim distribution:")
print(f"    min={min(cos_sims):.3f}  median={np.median(cos_sims):.3f}  max={max(cos_sims):.3f}")

# Parse success rate
parse_rate = len(predictions) / len(eval_dataset)
print(f"\n  Parse Success Rate:   {parse_rate:.3f}  ({len(predictions)}/{len(eval_dataset)})")
print(f"{'=' * 60}")

  MODEL 1 EVALUATION SUMMARY  —  125 samples
  Depth Accuracy:       0.592  (74/125)
  ROUGE-L:              0.292
  Cosine Similarity:    0.506
  BERTScore F1:         0.880

  Depth Accuracy by Class:
    depth=-1:  0.571  (8/14)
    depth= 0:  0.568  (50/88)
    depth=+1:  0.696  (16/23)

  ROUGE-L distribution:
    min=0.000  median=0.267  max=0.941
  Cosine Sim distribution:
    min=-0.025  median=0.517  max=0.989

  Parse Success Rate:   1.000  (125/125)


In [9]:
# Per-event breakdown
print("\nPer-event breakdown:")
print(f"{'Idx':>4}  {'GT Depth':>8}  {'Pred Depth':>10}  {'Match':>5}  {'ROUGE-L':>7}  {'CosSim':>6}")
print("-" * 55)
for (idx, gt, pred), rs, cs in zip(matched, rouge_scores, cos_sims):
    ok = "ok" if gt["depth"] == pred["depth"] else "--"
    print(f"{idx:>4}  {gt['depth']:>8}  {pred['depth']:>10}  {ok:>5}  {rs:>7.3f}  {cs:>6.3f}")

# Save full results to JSON
results = {
    "dataset":           HF_DATASET,
    "n_gt":              n_gt,
    "n_matched":         n_matched,
    "depth_accuracy":    depth_acc,
    "rouge_l":           mean_rouge,
    "cosine_similarity": mean_cos,
    "bertscore_f1":      mean_bert,
    "per_event": [
        {
            "idx":             idx,
            "gt_depth":        gt["depth"],
            "pred_depth":      pred["depth"],
            "gt_annotation":   gt["annotation"],
            "pred_annotation": pred["annotation"],
            "depth_correct":   gt["depth"] == pred["depth"],
            "rouge_l":         rs,
            "cosine_sim":      cs,
        }
        for (idx, gt, pred), rs, cs in zip(matched, rouge_scores, cos_sims)
    ],
}

with open(RESULTS_JSON, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nFull results saved to {RESULTS_JSON}")

from google.colab import files
files.download(RESULTS_JSON)


Per-event breakdown:
 Idx  GT Depth  Pred Depth  Match  ROUGE-L  CosSim
-------------------------------------------------------
   0         0          -1     --    0.143   0.298
   1         0           0     ok    0.133   0.395
   2         1           0     --    0.588   0.717
   3         0           0     ok    0.077   0.026
   4         0           0     ok    0.286   0.320
   5         0           0     ok    0.213   0.617
   6         0           0     ok    0.250   0.449
   7         1           1     ok    0.556   0.848
   8         0           1     --    0.000   0.068
   9        -1           0     --    0.133   0.422
  10         0          -1     --    0.323   0.385
  11         0           1     --    0.286   0.469
  12        -1           0     --    0.273   0.422
  13         1           1     ok    0.133   0.487
  14         0           0     ok    0.333   0.439
  15         0           0     ok    0.160   0.073
  16         0           0     ok    0.273   0.456
  17

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>